In [1]:
import logging
import os

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# local .py file
from scPRINT import (
    process_model,
    populate_lamin_db,
    SCPRINT_DEFS
)
from etl_utils import (
    load_results,
    RESULTS_DEFS
)
from analysis_utils import (
    compute_attention_from_weights
)


INFO:numexpr.utils:NumExpr defaulting to 16 threads.


→ connected lamindb: anonymous/scPRINT_lamin
No module named 'triton'
FlashAttention is not installed, not using it..


In [2]:
# Configuration
DATA_DIR = "data"
OUTPUT_DIR = "output"

MODEL_PATH = os.path.join(DATA_DIR, "scPRINT")
os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [3]:
# Get list of version keys from SCPRINT_DEFS
SCPRINT_VERSION_KEYS = list(SCPRINT_DEFS.VERSIONS.__dict__.keys())

In [4]:
populate_lamin_db()

for version in SCPRINT_VERSION_KEYS:
    process_model(version, OUTPUT_DIR, MODEL_PATH)

INFO:scPRINT:Lamin database already configured
INFO:scPRINT:Extracting: scPRINT small (SMALL)
INFO:scPRINT:
1. Downloading/loading model if needed...


small-v1.ckpt:   0%|          | 0.00/111M [00:00<?, ?B/s]

INFO:scPRINT:Loading scPRINT model
INFO:scPRINT:Loading scPRINT model
INFO:scPRINT:Loading gene annotations


RuntimeError caught: scPrint is not attached to a `Trainer`.


INFO:scPRINT:Formatting model metadata
INFO:scPRINT:   44756 genes, 4 layers
INFO:scPRINT:2. Extracting weights...
INFO:scPRINT:   Embeddings: (44756, 128)
INFO:scPRINT:   Attention weights: 4 layers × 4 matrices (Q,K,V,O)
INFO:scPRINT:3. Saving results...
INFO:etl_utils:Saving weights to output/scPRINT_small_weights.npz and metadata to output/scPRINT_small_metadata.json
INFO:etl_utils:Successfully validated weights, gene metadata and model metadata
INFO:etl_utils:Saving weights to output/scPRINT_small_weights.npz
INFO:etl_utils:Saving metadata to output/scPRINT_small_metadata.json
INFO:etl_utils:Successfully saved all results
INFO:scPRINT:   Successfully saved all results!
INFO:scPRINT:Extracting: scPRINT medium (MEDIUM)
INFO:scPRINT:
1. Downloading/loading model if needed...
INFO:scPRINT:Loading scPRINT model
INFO:scPRINT:Loading scPRINT model
INFO:scPRINT:Loading gene annotations


...

INFO:scPRINT:Formatting model metadata
INFO:scPRINT:   44756 genes, 8 layers
INFO:scPRINT:2. Extracting weights...
INFO:scPRINT:   Embeddings: (44756, 256)
INFO:scPRINT:   Attention weights: 8 layers × 4 matrices (Q,K,V,O)
INFO:scPRINT:3. Saving results...
INFO:etl_utils:Saving weights to output/scPRINT_medium_weights.npz and metadata to output/scPRINT_medium_metadata.json
INFO:etl_utils:Successfully validated weights, gene metadata and model metadata
INFO:etl_utils:Saving weights to output/scPRINT_medium_weights.npz
INFO:etl_utils:Saving metadata to output/scPRINT_medium_metadata.json
INFO:etl_utils:Successfully saved all results
INFO:scPRINT:   Successfully saved all results!
INFO:scPRINT:Extracting: scPRINT large (LARGE)
INFO:scPRINT:
1. Downloading/loading model if needed...


large-v1.ckpt:   0%|          | 0.00/1.24G [00:00<?, ?B/s]

INFO:scPRINT:Loading scPRINT model
INFO:scPRINT:Loading scPRINT model
INFO:scPRINT:Loading gene annotations


RuntimeError caught: scPrint is not attached to a `Trainer`.


INFO:scPRINT:Formatting model metadata
INFO:scPRINT:   44756 genes, 16 layers
INFO:scPRINT:2. Extracting weights...
INFO:scPRINT:   Embeddings: (44756, 512)
INFO:scPRINT:   Attention weights: 16 layers × 4 matrices (Q,K,V,O)
INFO:scPRINT:3. Saving results...
INFO:etl_utils:Saving weights to output/scPRINT_large_weights.npz and metadata to output/scPRINT_large_metadata.json
INFO:etl_utils:Successfully validated weights, gene metadata and model metadata
INFO:etl_utils:Saving weights to output/scPRINT_large_weights.npz
INFO:etl_utils:Saving metadata to output/scPRINT_large_metadata.json
INFO:etl_utils:Successfully saved all results
INFO:scPRINT:   Successfully saved all results!


In [5]:
# Load results for a specific version (using MEDIUM as an example)
from etl_utils import MODELS
medium_version_id = SCPRINT_DEFS.VERSIONS.MEDIUM
file_prefix = f"{MODELS.SCPRINT}_{medium_version_id}"
weights_dict, gene_annotations, model_metadata = load_results(OUTPUT_DIR, file_prefix)

GENES_OF_INTEREST = gene_annotations[RESULTS_DEFS.VOCAB_NAME].sample(20000).tolist()
GENE_MASK = [x in GENES_OF_INTEREST for x in model_metadata[RESULTS_DEFS.ORDERED_VOCABULARY]]

# Compute attention on demand
layer_attn = compute_attention_from_weights(
    weights_dict[RESULTS_DEFS.GENE_EMBEDDING][GENE_MASK,:],
    weights_dict[RESULTS_DEFS.ATTENTION_WEIGHTS]['layer_5'][RESULTS_DEFS.W_Q],
    weights_dict[RESULTS_DEFS.ATTENTION_WEIGHTS]['layer_5'][RESULTS_DEFS.W_K]
)

INFO:etl_utils:Loading weights from output/scPRINT_medium_weights.npz and metadata from output/scPRINT_medium_metadata.json
INFO:etl_utils:Loading weights from output/scPRINT_medium_weights.npz
INFO:etl_utils:Loading metadata from output/scPRINT_medium_metadata.json
INFO:etl_utils:Successfully loaded and validated all results
